# Daily Challenge — Fine-tuning BERT & XLM-RoBERTa for Text Classification

**Dataset:** Multilingual Natural Language Inference (NLI)  
**Task:** Classify the relationship between a *premise* and a *hypothesis* into 3 labels:
- `0` → Entailment
- `1` → Neutral  
- `2` → Contradiction

---

## 0. Install Dependencies

In [ ]:
!pip install transformers datasets scikit-learn pandas torch -q

## 0.1 Upload the Dataset

Upload `train.csv` and `test.csv` from your local machine.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select train.csv and test.csv

---
## Part 1 — Understanding BERT and XLM-RoBERTa

### What is BERT?
**BERT** (Bidirectional Encoder Representations from Transformers) is a pre-trained transformer model developed by Google.
- It reads text **bidirectionally** (left-to-right AND right-to-left simultaneously).
- Pre-trained on English text using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
- Best suited for **English-only** tasks.

### What is XLM-RoBERTa?
**XLM-RoBERTa** (Cross-lingual Language Model - RoBERTa) is a multilingual model developed by Facebook AI.
- Pre-trained on **100 languages** — perfect for our multilingual NLI dataset.
- Based on RoBERTa (an optimized version of BERT without NSP, trained longer).
- Uses **SentencePiece** tokenization, which handles any language script (Arabic, Chinese, Urdu…).

| Feature | BERT | XLM-RoBERTa |
|---|---|---|
| Languages | English (mainly) | 100 languages |
| Tokenizer | WordPiece | SentencePiece |
| Special tokens | [CLS], [SEP], [PAD] | `<s>`, `</s>`, `<pad>` |
| Best for | English NLP | Multilingual NLP |

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Load pre-trained tokenizers
bert_tokenizer    = BertTokenizer.from_pretrained('bert-base-uncased')
xlmr_tokenizer    = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

print(f"BERT vocabulary size  : {bert_tokenizer.vocab_size:,}")
print(f"XLM-R vocabulary size : {xlmr_tokenizer.vocab_size:,}")

print("\nBERT special tokens  :", bert_tokenizer.special_tokens_map)
print("XLM-R special tokens :", xlmr_tokenizer.special_tokens_map)

---
## Part 2 — Tokenizing Text

Tokenization converts raw text into numerical IDs the model can process.

**Key output fields:**
- `input_ids` — token IDs (integers)
- `attention_mask` — 1 for real tokens, 0 for padding
- `token_type_ids` — 0 for sentence A, 1 for sentence B (BERT only)

In [ ]:
# Example sentences from our NLI dataset
premise    = "These comments were considered when formulating the interim rules."
hypothesis = "The rules were developed with these comments in mind."

# ── BERT tokenization ──────────────────────────────────────────────────────
# encode_plus() est déprécié — on appelle le tokenizer directement
bert_encoding = bert_tokenizer(
    premise,
    hypothesis,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print("=== BERT ===")
print("input_ids     :", bert_encoding['input_ids'])
print("attention_mask:", bert_encoding['attention_mask'])
print("token_type_ids:", bert_encoding['token_type_ids'])
print("Decoded       :", bert_tokenizer.decode(bert_encoding['input_ids'][0]))

In [ ]:
# ── XLM-RoBERTa tokenization ───────────────────────────────────────────────
# Note: XLM-R does NOT produce token_type_ids
xlmr_encoding = xlmr_tokenizer(
    premise,
    hypothesis,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print("=== XLM-RoBERTa ===")
print("input_ids     :", xlmr_encoding['input_ids'])
print("attention_mask:", xlmr_encoding['attention_mask'])
print("Decoded       :", xlmr_tokenizer.decode(xlmr_encoding['input_ids'][0]))

In [ ]:
# Experiment: multilingual sentence (French from our dataset)
fr_premise    = "Des petites choses comme celles-là font une différence énorme."
fr_hypothesis = "J'essayais d'accomplir quelque chose."

for name, tokenizer in [('BERT', bert_tokenizer), ('XLM-R', xlmr_tokenizer)]:
    tokens = tokenizer.tokenize(fr_premise)
    print(f"{name} tokens: {tokens}")

---
## Part 3 — Preparing Input Data for the Model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 128

class NLIDataset(Dataset):
    """
    PyTorch Dataset for Natural Language Inference.
    Tokenizes (premise, hypothesis) pairs and returns model-ready tensors.
    """

    def __init__(self, premises, hypotheses, labels, tokenizer, max_len=MAX_LEN):
        self.premises    = premises
        self.hypotheses  = hypotheses
        self.labels      = labels
        self.tokenizer   = tokenizer
        self.max_len     = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Quick demo with 3 examples
demo_premises    = ["The sky is blue.", "It is raining.",  "Cats are pets."]
demo_hypotheses  = ["The sky has color.", "It is sunny.", "Cats are wild animals."]
demo_labels      = [0, 2, 2]

demo_dataset = NLIDataset(demo_premises, demo_hypotheses, demo_labels, xlmr_tokenizer)
sample = demo_dataset[0]

print("input_ids shape     :", sample['input_ids'].shape)
print("attention_mask shape:", sample['attention_mask'].shape)
print("label               :", sample['labels'].item())
print("Non-padding tokens  :", sample['attention_mask'].sum().item(), "/ 128")
print("\nSpecial tokens (XLM-R):", xlmr_tokenizer.special_tokens_map)
print("Vocab size (XLM-R)    :", xlmr_tokenizer.vocab_size)

---
## Part 4 — Loading and Exploring the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load CSVs
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print("Train shape:", train_df.shape)
print("Test  shape:", test_df.shape)

In [ ]:
train_df.head(10)

In [ ]:
test_df.head(5)

In [ ]:
print("Column types:")
print(train_df.dtypes)
print("\nNull values:")
print(train_df.isnull().sum())

In [ ]:
# Label distribution
label_map = {0: 'Entailment', 1: 'Neutral', 2: 'Contradiction'}
train_df['label_name'] = train_df['label'].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label counts
train_df['label_name'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#3498db','#e74c3c'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Language distribution
train_df['language'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Language Distribution')
axes[1].set_xlabel('Language')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(train_df['label_name'].value_counts())

In [ ]:
# Text length analysis
train_df['premise_len']    = train_df['premise'].str.split().str.len()
train_df['hypothesis_len'] = train_df['hypothesis'].str.split().str.len()

print("Premise    — mean length:",   round(train_df['premise_len'].mean(), 1),
      "| max:", train_df['premise_len'].max())
print("Hypothesis — mean length:",   round(train_df['hypothesis_len'].mean(), 1),
      "| max:", train_df['hypothesis_len'].max())

---
## Part 5 — Creating Cross-Validation Folds

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
RANDOM_STATE = 42

X = train_df[['premise', 'hypothesis']].values
y = train_df['label'].values

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

train_folds = []
val_folds   = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    train_folds.append(train_df.iloc[train_idx].reset_index(drop=True))
    val_folds.append(train_df.iloc[val_idx].reset_index(drop=True))

    print(f"Fold {fold+1}: train={len(train_idx):,}  val={len(val_idx):,}")
    for lbl, name in label_map.items():
        tr_pct  = (train_df.iloc[train_idx]['label'] == lbl).mean() * 100
        val_pct = (train_df.iloc[val_idx]['label']   == lbl).mean() * 100
        print(f"   {name:>15}: train {tr_pct:.1f}%  |  val {val_pct:.1f}%")
    print()

In [ ]:
# Visualize fold balance
fig, axes = plt.subplots(1, N_SPLITS, figsize=(16, 4), sharey=True)

for fold in range(N_SPLITS):
    val_folds[fold]['label_name'].value_counts().plot(
        kind='bar', ax=axes[fold],
        color=['#2ecc71','#3498db','#e74c3c']
    )
    axes[fold].set_title(f'Fold {fold+1} — Validation')
    axes[fold].tick_params(axis='x', rotation=30)

plt.suptitle('Label Distribution per Validation Fold', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 6 — Building and Fine-tuning XLM-RoBERTa

We use **XLM-RoBERTa** because the dataset is multilingual (English, French, Arabic, Chinese, Urdu…).

In [ ]:
import torch
from transformers import (
    XLMRobertaForSequenceClassification,
    XLMRobertaTokenizer,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_LABELS = 3
BATCH_SIZE = 16
EPOCHS     = 2       # Increase to 3-5 for better performance
LR         = 2e-5
MAX_LEN    = 128

print("Device:", DEVICE)

In [ ]:
# ── Training function ──────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


# ── Evaluation function ────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

In [ ]:
# ── Train on Fold 1 only (demonstration) ──────────────────────────────────
# To run full cross-validation, wrap the code below in: for fold in range(N_SPLITS):

FOLD = 0  # Change to loop over all folds for full CV

fold_train = train_folds[FOLD]
fold_val   = val_folds[FOLD]

tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

train_dataset = NLIDataset(
    fold_train['premise'].tolist(),
    fold_train['hypothesis'].tolist(),
    fold_train['label'].tolist(),
    tokenizer,
    MAX_LEN
)
val_dataset = NLIDataset(
    fold_val['premise'].tolist(),
    fold_val['hypothesis'].tolist(),
    fold_val['label'].tolist(),
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Fold {FOLD+1} — train batches: {len(train_loader)} | val batches: {len(val_loader)}")

In [ ]:
# Load the model
model = XLMRobertaForSequenceClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=NUM_LABELS
)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    print(f"\n── Epoch {epoch+1}/{EPOCHS} ──")
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler)
    vl_loss, vl_acc, preds, labels = evaluate(model, val_loader)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    print(f"Train  → loss: {tr_loss:.4f}  acc: {tr_acc:.4f}")
    print(f"Val    → loss: {vl_loss:.4f}  acc: {vl_acc:.4f}")

print("\n", classification_report(labels, preds, target_names=list(label_map.values())))

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'],   label='Val Loss',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'],   label='Val Acc',   marker='o')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('XLM-RoBERTa — Training Curves (Fold 1)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 7 — Generate Predictions on Test Set

In [ ]:
class NLITestDataset(Dataset):
    def __init__(self, premises, hypotheses, tokenizer, max_len=MAX_LEN):
        self.premises   = premises
        self.hypotheses = hypotheses
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }


test_dataset = NLITestDataset(
    test_df['premise'].tolist(),
    test_df['hypothesis'].tolist(),
    tokenizer,
    MAX_LEN
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
all_predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_predictions.extend(preds)

submission = pd.DataFrame({
    'id':         test_df['id'],
    'prediction': all_predictions
})

submission.to_csv('submission.csv', index=False)
print("submission.csv saved!")
submission.head(10)

In [ ]:
# Download submission file
from google.colab import files
files.download('submission.csv')

---
## Summary

| Part | What we did |
|------|-------------|
| 1 | Compared BERT vs XLM-RoBERTa — chose XLM-R for multilingual data |
| 2 | Tokenized (premise, hypothesis) pairs with `encode_plus()` |
| 3 | Built `NLIDataset` — padded to 128 tokens, exported `input_ids` + `attention_mask` |
| 4 | Loaded CSVs, explored label & language distribution |
| 5 | Created 5 stratified folds — each fold preserves label proportions |
| 6 | Fine-tuned `xlm-roberta-base` with AdamW + linear warmup |
| 7 | Generated `submission.csv` from test set predictions |